# Deal Prioritizer: API walkthrough (local)

Given a trade and a city, the API pulls local businesses from OpenStreetMap, drops the chains, scores the rest on what their listings actually say, and exports a shortlist you can start calling.

This one runs against the Docker stack on your machine. Start it with `docker compose up --build`, then run the cells top to bottom. The only dependency is `httpx` (`pip install httpx`). Swagger for every endpoint is at http://localhost:8000/docs.

For the deployed version, see [`api_walkthrough_live.ipynb`](api_walkthrough_live.ipynb).

Business data © OpenStreetMap contributors, ODbL.

In [1]:
import csv
import io
from collections import Counter

import httpx

API = "http://localhost:8000"
client = httpx.Client(base_url=API, timeout=120)
print(client.get("/healthz").json())

{'ok': True, 'app': 'deal-prioritizer'}


## 1. Run a search

One request does the whole thing: geocode the market, pull matching businesses from OpenStreetMap, dedupe, score, and save the run in Postgres. If you run the same search again within the hour, it comes from the cache.

In [2]:
res = client.post(
    "/v1/pipeline/runs",
    json={"vertical": "Auto", "market": "Chicago, IL", "limit": 35},
)
run = res.json()
print(res.status_code, "| source:", run["source_status"], "|", len(run["targets"]), "companies")
print("detail:", run["source_detail"] or "none")

201 | source: live | 34 companies
detail: none


## 2. The ranked shortlist

Every score comes with the reasons behind it, so whoever makes the calls can see why a shop ranks where it does.

In [3]:
shortlist = [t for t in run["targets"] if not t["skipped"]][:10]
for t in shortlist:
    contact = t["main_phone"] or t["email"] or "-"
    print(f"{t['fit_score']:>5} {t['tier']:<6} {t['legal_name'][:32]:<33} {contact:<16} {t['rationale']}")

 87.0 prime  Lara Auto Service 2               +1-773-585-9040  Phone listed · Street address · Website listed · Founded 2004 (~22y)
 75.0 solid  Erie LaSalle Body Shop            +1-312-337-3903  Phone listed · Street address · Website listed
 75.0 solid  Rojas Auto Service                +1-773-247-9318  Phone listed · Street address · Website listed
 70.0 solid  Hawk Motors                       +1 773-248-1200  Phone listed · Street address
 70.0 solid  I & C Auto Repair                 +1 773-549-8254  Phone listed · Street address
 63.0 solid  Fulton-Desplaines Garage          -                Street address · Website listed
 58.0 solid  Nortown                           -                Street address · No contact details in OSM, enrich first
 58.0 solid  Chicago Auto Repair Service Inc.  -                Street address · No contact details in OSM, enrich first
 58.0 solid  Tony's Auto Center                -                Street address · No contact details in OSM, enrich firs

## 3. Chains get filtered out

OpenStreetMap tags franchise locations with `brand:wikidata`, which is enough to reject the national chains before anyone spends a call on them.

In [4]:
for t in run["targets"]:
    if t["skipped"]:
        print(f"{t['legal_name']:<28} {t['rationale']}")

Car-X                        Chain or franchise (Car-X)
Rivian Service Center        Chain or franchise (Rivian)
Goodyear                     Chain or franchise (Goodyear)
Gerber Collision & Glass     Chain or franchise (Gerber Collision & Glass)
Caliber Collision            Chain or franchise (Caliber Collision)
Crash Champions              Chain or franchise (Crash Champions)


## 4. Tier breakdown

In [5]:
print(Counter(t["tier"] for t in run["targets"]))

Counter({'watch': 18, 'solid': 9, 'reject': 6, 'prime': 1})


## 5. Export for outreach

Same filters as the UI: companies scoring 58 or higher, no chains. The CSV imports straight into a CRM or dialer.

In [6]:
export = client.get(f"/v1/pipeline/runs/{run['id']}/export", params={"min_score": 58})
rows = list(csv.DictReader(io.StringIO(export.text)))
print(len(rows), "rows | columns:", ", ".join(rows[0].keys()))
for r in rows[:3]:
    print(" ", r["fit_score"], r["company"], r["phone"], r["website"])

10 rows | columns: fit_score, tier, company, phone, email, website, address, city, state, notes, vertical, market
  87.0 Lara Auto Service 2 +1-773-585-9040 http://laraautoservice2.com
  75.0 Rojas Auto Service +1-773-247-9318 https://www.rojasautoservice.com/
  75.0 Erie LaSalle Body Shop +1-312-337-3903 https://erielasalle.com


## 6. Validation

Pydantic rejects bad input before any work happens, and a market the geocoder can't find gets a clear error rather than made-up results.

In [7]:
bad_band = client.post(
    "/v1/pipeline/runs",
    json={"vertical": "Auto", "market": "Chicago, IL", "headcount_min": 50, "headcount_max": 10},
)
print(bad_band.status_code, bad_band.json()["detail"][0]["msg"])

unknown = client.post("/v1/pipeline/runs", json={"vertical": "Auto", "market": "Nowhereville, ZZ"})
print(unknown.status_code, unknown.json()["detail"])

422 headcount_min cannot exceed headcount_max
422 Couldn't find the market 'Nowhereville, ZZ'. Try 'City, ST'.
